## 1. 지역별 데이터 불러오기

In [ ]:
import pandas as pd
import os
import re

# 폴더 경로 설정
folder_path = '../Database/opendata'

# csv 파일 필터링
file_list = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# 데이터프레임 리스트
df_list = []

for file in file_list:
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(file_path, encoding='utf-8-sig') 
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='euc-kr')  
    df_list.append(df)

# 모든 데이터를 하나로 합치기
all_data = pd.concat(df_list, ignore_index=True)

# 미리보기
all_data.head()


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_115184\2200916044.py:17: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8')  # 기본 utf-8
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_115184\2200916044.py:17: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8')  # 기본 utf-8
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_115184\2200916044.py:17: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8')  # 기본 utf-8
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_115184\2200916044.py:17: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8')  # 기본 utf-8
C:\Users\Public\Documents\ESTsof

,statNm,statId,chgerId,chgerType,addr,addrDetail,location,useTime,lat,lng,...,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName
0,원주(부산) 휴게소,ME178073,1,6,강원특별자치도 원주시 호저면 마근거리길 120 (옥산리) 211,NaN,NaN,24시간 이용가능,37.434578,127.929205,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
1,원주(부산) 휴게소,ME178073,2,6,강원특별자치도 원주시 호저면 마근거리길 120 (옥산리) 211,NaN,NaN,24시간 이용가능,37.434578,127.929205,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
2,원주(부산) 휴게소,ME178073,3,6,강원특별자치도 원주시 호저면 마근거리길 120 (옥산리) 211,NaN,NaN,24시간 이용가능,37.434578,127.929205,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
3,치악(부산) 휴게소,ME178077,1,6,강원특별자치도 원주시 신림면 치악로 416 (금창리),NaN,NaN,24시간 이용가능,37.253311,128.049451,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
4,치악(부산) 휴게소,ME178077,2,6,강원특별자치도 원주시 신림면 치악로 416 (금창리),NaN,NaN,24시간 이용가능,37.253311,128.049451,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도


In [ ]:
# 총 데이터 개수 확인: (412178, 35)

all_data.shape

(412178, 35)

In [75]:
col_names = all_data.columns

## 2. 데이터 전처리

1. delYn 변수에서 n인 것들만 사용
2. kindDetail에서 아파트 등 접근 불가한 데이터 지우기
3. 24시간인지 아닌지 컬럼 전처리 -> 1차 개발시에만 전처리 이슈로 이렇게 하고 나중에 고도화
4. 시간 관련 column 데이터타입 변경

In [82]:
i = 30
print(col_names[i], all_data[col_names[i]].unique())

delYn ['N' 'Y']


In [90]:
# 삭제된 충전소(운영하지 않는 충전소) 제거
all_data_2 = all_data[all_data['delYn']=='N']

In [ ]:
# 제거 후 개수 확인 
all_data_2.shape

(412055, 35)

In [ ]:
# kindDetail 기반 접근 불가 충전기 제거 

all_data_3 = all_data_2[~all_data_2['kindDetail'].isin(['G001', 'G005', 'G006', 'H001', 'H002', 'H003', 'H004', 'H005'])]

In [ ]:
# 제거 후 개수 확인 

all_data_3.shape

(116503, 35)

In [ ]:
# 24시간 운영하는 충전소만 필터링

all_data_4 = all_data_3[all_data_3['useTime'].str.contains('24시간', na=False)]

In [ ]:
# 제거 후 개수 확인 

all_data_4.shape

(91466, 35)

In [ ]:
'''
운영시간 칼럼 너무 복잡해서 나중에 노가다로 해야할듯
'''

# 요일 매핑 딕셔너리
# weekday_map = {
#     '월': 'Mon', '화': 'Tue', '수': 'Wed', '목': 'Thu', '금': 'Fri',
#     '토': 'Sat', '일': 'Sun',
#     '평일': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri'],
#     '주말': ['Sat', 'Sun'],
#     '매일': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
#     '주중': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri'],
#     'All': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
# }

# def extract_schedule(ustime, statID):
#     result = []

#     if pd.isna(ustime):
#         return []

#     # 24시간 단순표현
#     if '24시간' in ustime or re.search(r'00:00\s*~\s*24:00', ustime):
#         for day in weekday_map['All']:
#             result.append({'statID': statID, 'day': day, 'start_time': '00:00', 'end_time': '24:00'})
#         return result

#     # 시간 범위 추출 (e.g., 09:00~18:00)
#     pattern = re.findall(r'([가-힣]+)?\s*:?(\d{1,2}[:시]?\d{0,2})\s*[-~]\s*(\d{1,2}[:시]?\d{0,2})', ustime)
#     for group in pattern:
#         raw_day, start_raw, end_raw = group
#         # 시간 형식 보정
#         def normalize(t):
#             t = t.replace('시', ':00').replace(' ', '')
#             if ':' not in t:
#                 t += ':00'
#             if len(t.split(':')[0]) == 1:
#                 t = '0' + t
#             return t

#         start = normalize(start_raw)
#         end = normalize(end_raw)

#         # 요일 처리
#         if raw_day:
#             days = weekday_map.get(raw_day.strip(), [])
#             if isinstance(days, str): days = [days]
#         else:
#             days = weekday_map['All']

#         for day in days:
#             result.append({'statID': statID, 'day': day, 'start_time': start, 'end_time': end})

#     return result

# df: 원본 DataFrame (ustime 포함, statID 포함)
# rows = []

# for i, row in all_data.iterrows():
#     parsed = extract_schedule(row['useTime'], row['statId'])
#     rows.extend(parsed)

# # DataFrame으로 변환
# schedule_df = pd.DataFrame(rows)

# 저장
# schedule_df.to_csv('station_hours.csv', index=False)\


''

In [ ]:
all_data_4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 91466 entries, 0 to 412177
Data columns (total 35 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   statNm       91466 non-null  object 
 1   statId       91466 non-null  object 
 2   chgerId      91466 non-null  int64  
 3   chgerType    91466 non-null  int64  
 4   addr         91466 non-null  object 
 5   addrDetail   38583 non-null  object 
 6   location     58697 non-null  object 
 7   useTime      91466 non-null  object 
 8   lat          91466 non-null  float64
 9   lng          91466 non-null  float64
 10  busiId       91466 non-null  object 
 11  bnm          91466 non-null  object 
 12  busiNm       91466 non-null  object 
 13  busiCall     91317 non-null  object 
 14  stat         91466 non-null  int64  
 15  statUpdDt    88320 non-null  float64
 16  lastTsdt     86018 non-null  float64
 17  lastTedt     86118 non-null  float64
 18  nowTsdt      9825 non-null   float64
 19  powerTyp

In [ ]:
# 시간 관련 칼럼 데이터타입 변경 

for col in ['statUpdDt', 'lastTsdt', 'lastTedt', 'nowTsdt']:
    all_data_4.loc[:, col] = pd.to_datetime(all_data_4[col], format='%Y%m%d%H%M%S', errors='coerce')

In [ ]:
# 결과 확인 

all_data_4.iloc[:, 10:].head(5)

,busiId,bnm,busiNm,busiCall,stat,statUpdDt,lastTsdt,lastTedt,nowTsdt,powerType,...,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName
0,ME,환경부,환경부,1661-9408,2,2025-05-15 15:55:32,2025-05-15 14:53:26,2025-05-15 15:07:11,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
1,ME,환경부,환경부,1661-9408,2,2025-05-15 15:56:33,2025-05-15 13:33:14,2025-05-15 14:05:38,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
2,ME,환경부,환경부,1661-9408,2,2025-05-15 15:54:34,2025-05-14 19:45:11,2025-05-14 20:26:46,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
3,ME,환경부,환경부,1661-9408,2,2025-05-15 15:57:34,2025-05-15 11:15:42,2025-05-15 11:46:57,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
4,ME,환경부,환경부,1661-9408,2,2025-05-15 15:57:35,2025-05-14 11:13:52,2025-05-14 11:59:58,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도


## 3. 로직 1 - 1차 필터링

로직 1: 사용자 입력 변수를 기반으로 한 필터링

1차 필터링: 고장나지 않은 충전기만 필터링
- 이용자제한 여부 -> limitYn 기준으로 이용자 제한 시설 제외 -> 고민 필요
- stat 변수에서 1, 2, 3, 9번만 사용 (1: 통신이상, 2: 충전대기, 3: 충전중, 9: 상태미확인)
- 고장/삭제 여부: 최근 사용이 현재 시점으로부터 48시간 이내에 발생한 적이 있는 충전소는 정상 충전소로 간주 

In [145]:
# limitYn으로 거르기엔 그냥 단순 시간 제한이 사유인 곳도 많아서 고민. 제한 사유가 '제한 없음' 인 곳도 있고..

all_data_4[all_data_4['limitYn']=='Y']['limitDetail'].unique()[43:60]

array(['버스전용', '제한없음', '제한 없음', '.', '마트(쇼핑몰) 이용자, 상가 입주자로 사용 제한',
       '택시차고지 차량이 다수로 이용이 어려울수 있음', '전기버스 전용(DC콤보2)',
       '시설 사용자 외 이용제한 있을 수 있음', '외부인이 전기차 충전목적만으로 출입할 수 없습니다', '외부인 출입불가',
       '관용차량 충전 전용', '구내시설 상황에 따라 이용이 제한될 수 있음',
       '시설 상황에 따라 이용이 제한될 수 있음,', '학교 교직원으로 제한', '숙박객외 사용불가',
       '매장/시설 이용고객만 사용가능', '직원 및 방문자 전용'], dtype=object)

In [ ]:
# stat이 1인 경우에도 정상 작동하는 경우가 있는지 확인 -> 일단 48시간 이내에 사용된 기록이 있는 곳도 있긴 해서 포함. 

all_data_4[all_data_4['stat']==1].iloc[:, 10:].head(5)

,busiId,bnm,busiNm,busiCall,stat,statUpdDt,lastTsdt,lastTedt,nowTsdt,powerType,...,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName
874,CG,서울씨엔지,서울씨엔지(서울이브이),1566-6373,1,2025-05-14 23:47:03,2025-05-14 21:50:31,2025-05-14 23:10:22,NaT,NaN,...,J007,Y,NaN,N,NaN,N,NaN,N,2023,강원특별자치도
875,CG,서울씨엔지,서울씨엔지(서울이브이),1566-6373,1,2025-05-14 23:47:03,2025-05-14 21:50:31,2025-05-14 23:10:22,NaT,NaN,...,J007,Y,NaN,N,NaN,N,NaN,N,2023,강원특별자치도
958,CI,쿨사인,쿨사인 주식회사,1600-4045,1,2025-04-30 06:59:41,2024-11-11 17:24:35,2024-11-11 17:30:02,NaT,NaN,...,G004,Y,NaN,N,NaN,N,NaN,N,2024,강원특별자치도
959,CI,쿨사인,쿨사인,1600-4045,1,2025-05-07 09:30:04,2025-05-06 18:02:18,2025-05-07 09:28:26,NaT,NaN,...,G004,Y,NaN,N,NaN,N,NaN,N,2024,강원특별자치도
978,CI,쿨사인,쿨사인,1600-4045,1,2025-04-30 02:46:42,2024-08-17 14:55:29,2024-08-17 16:27:15,NaT,NaN,...,G004,Y,NaN,N,NaN,N,NaN,N,2024,강원특별자치도


In [ ]:
# stat 칼럼 기반 필터링 함수 정의

def stat_filtering(df):
    return df[df['stat'].isin([1, 2, 3, 9])]

stat_filtering(all_data_4).shape

(90891, 35)

In [ ]:

# 현재 시점 기준 48시간 이내에 사용 기록이 있는지 여부 기반 필터링 함수 정의 

def recent_filtering(df, hours=48, now=None):
    
    if now is None:
        now = pd.Timestamp.now()
    
    # 최근 충전 '시작' 시간과 최근 충전 '종료' 시간 중 더 최근 시간을 기준으로 계산
    max_time = df[['lastTsdt', 'lastTedt']].max(axis=1)

    # 기준 시각에서 `hours` 이전보다 더 늦은 것만 남기기
    return df[max_time >= (now - pd.Timedelta(hours=hours))].copy()

In [146]:
# 테스트트

from datetime import datetime
custom_now = pd.Timestamp(datetime(2025, 5, 16, 12, 0, 0))

recent_filtering(all_data_4, 48, now=custom_now).head(5).iloc[:, 10:]

,busiId,bnm,busiNm,busiCall,stat,statUpdDt,lastTsdt,lastTedt,nowTsdt,powerType,...,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName
0,ME,환경부,환경부,1661-9408,2,2025-05-15 15:55:32,2025-05-15 14:53:26,2025-05-15 15:07:11,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
1,ME,환경부,환경부,1661-9408,2,2025-05-15 15:56:33,2025-05-15 13:33:14,2025-05-15 14:05:38,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
2,ME,환경부,환경부,1661-9408,2,2025-05-15 15:54:34,2025-05-14 19:45:11,2025-05-14 20:26:46,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
3,ME,환경부,환경부,1661-9408,2,2025-05-15 15:57:34,2025-05-15 11:15:42,2025-05-15 11:46:57,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
5,ME,환경부,환경부,1661-9408,2,2025-05-15 15:55:47,2025-05-14 20:58:55,2025-05-14 21:13:56,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도


In [ ]:
# 필터링 함수 통합한 최종 1차 필터링 함수 정의 

def filtering_first(df, hours=48, now=None):
    temp_df = stat_filtering(df)
    result_df = recent_filtering(temp_df, hours, now)
    return result_df

In [130]:
filtering_first(all_data_4, 48, custom_now).shape

(38338, 35)

## 4. m km 기반 필터링

In [ ]:
# haversine 라이브러리를 이용해 계산 가능하긴 함. 그러나 계산 시간 이슈로 numpy 기반 구현 예정. 

# from haversine import haversine, Unit
# import pandas as pd


# def filter_by_distance(df, center=(37.5665, 126.9780), max_distance_km=5):
    
#     def calc_distance(row):
#         point = (row['lat'], row['lng'])
#         return haversine(center, point, unit=Unit.KILOMETERS)

#     df = df.copy()
#     df['distance_km'] = df.apply(calc_distance, axis=1)
#     return df[df['distance_km'] <= max_distance_km]


In [148]:
import numpy as np

def filter_by_distance_vectorized(df, center, max_distance_km=5):
    R = 6371  # 지구 반지름 (단위: km)
    lat1, lon1 = np.radians(center)

    lat2 = np.radians(df['lat'].values)
    lon2 = np.radians(df['lng'].values)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distances = R * c

    return df.loc[distances <= max_distance_km].copy()


In [156]:
filter_by_distance_vectorized(all_data_4, (36, 129), 5).head(5)

,statNm,statId,chgerId,chgerType,addr,addrDetail,location,useTime,lat,lng,...,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName
164968,고경면사무소,ME18C157,1,6,경상북도 영천시 고경면 호국로 1065-4,NaN,NaN,24시간 이용가능,36.001288,129.046173,...,A002,Y,NaN,N,NaN,N,NaN,N,2018,경상북도
165270,고경면행정복지센터앞 공원 주차장,ME19F229,1,4,경상북도 영천시 고경면 호국로 1065-4,경상북도 영천시 고경면 호국로 1065-4,NaN,24시간 이용가능,36.001231,129.046306,...,B002,Y,NaN,N,NaN,N,NaN,N,2021,경상북도
165271,고경면행정복지센터앞 공원 주차장,ME19F229,2,4,경상북도 영천시 고경면 호국로 1065-4,경상북도 영천시 고경면 호국로 1065-4,NaN,24시간 이용가능,36.001231,129.046306,...,B002,Y,NaN,N,NaN,N,NaN,N,2019,경상북도
165812,망정2공영주차장,ME23A405,21,4,경상북도 영천시 망정동 417-21,NaN,NaN,24시간 이용가능,35.987394,128.957118,...,B001,Y,NaN,N,NaN,N,NaN,N,2023,경상북도
165813,망정2공영주차장,ME23A405,22,4,경상북도 영천시 망정동 417-21,NaN,NaN,24시간 이용가능,35.987394,128.957118,...,B001,Y,NaN,N,NaN,N,NaN,N,2023,경상북도


## 5. 로직 1 - 2차 필터링

2차 필터링: 사용자 입력 변수 기반 필터링
- output: 충전 용량(3, 7, 50, 100, 200)
- chgertype: 충전기 커넥터 유형(01:DC차데모,02: AC완속,03: DC차데모+AC3상,04: DC콤보,05: DC차데모+DC콤보, 06: DC차데모+AC3상+DC콤보, 07: AC3상, 08: DC콤보(완속), 09: NACS, 10: DC콤보+NACS)
- kind: 관련 시설 종류(공공시설, 주차시설 등)
- busid: 충전 사업자(GS 칼텍스, 현대자동차 등)

In [157]:
def filter_by_user_input(df, 
                         output_values=None, 
                         chger_types=None, 
                         kinds=None, 
                         busi_ids=None):
    filtered = df.copy()

    if output_values is not None:
        filtered = filtered[filtered['output'].isin(output_values)]
    
    if chger_types is not None:
        filtered = filtered[filtered['chgerType'].astype(str).isin(chger_types)]

    if kinds is not None:
        filtered = filtered[filtered['kind'].isin(kinds)]
    
    if busi_ids is not None:
        filtered = filtered[filtered['busiId'].isin(busi_ids)]

    return filtered


In [162]:
filter_by_user_input(
    df=all_data_4,
    output_values=[50, 100],
    busi_ids=['ME', 'GS']
).head(5)


,statNm,statId,chgerId,chgerType,addr,addrDetail,location,useTime,lat,lng,...,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName
0,원주(부산) 휴게소,ME178073,1,6,강원특별자치도 원주시 호저면 마근거리길 120 (옥산리) 211,NaN,NaN,24시간 이용가능,37.434578,127.929205,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
1,원주(부산) 휴게소,ME178073,2,6,강원특별자치도 원주시 호저면 마근거리길 120 (옥산리) 211,NaN,NaN,24시간 이용가능,37.434578,127.929205,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
2,원주(부산) 휴게소,ME178073,3,6,강원특별자치도 원주시 호저면 마근거리길 120 (옥산리) 211,NaN,NaN,24시간 이용가능,37.434578,127.929205,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
3,치악(부산) 휴게소,ME178077,1,6,강원특별자치도 원주시 신림면 치악로 416 (금창리),NaN,NaN,24시간 이용가능,37.253311,128.049451,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
4,치악(부산) 휴게소,ME178077,2,6,강원특별자치도 원주시 신림면 치악로 416 (금창리),NaN,NaN,24시간 이용가능,37.253311,128.049451,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
